In [ ]:
# Celda 1 - Imports y configuración
import numpy as np
import matplotlib.pyplot as plt
import joblib
import polars as pl
from pathlib import Path

# Rutas (MODIFICA ESTAS RUTAS SEGÚN TUS ARCHIVOS)
RUTA_MODELO_LGBM = "modelos/lightgbm_model.joblib"      # o .pkl / .txt
RUTA_MODELO_MLP  = "modelos/mlp_model.joblib"           # o .h5 / .keras
RUTA_DATASET     = "data/df_test.parquet"               # o .csv / .feather

# Parámetros de backtesting
ts = 0.55
fee = 0.00005
k = 100          # horizonte de retorno futuro

In [ ]:
# Celda 2 - Cargar dataset
print("Cargando dataset...")
df_test = pl.read_parquet(RUTA_DATASET)   # cambia a pl.read_csv() si es CSV

# Ajusta el nombre de la columna de precio si es diferente
columna_precio = "v1"
precios_completos = df_test.select(columna_precio).to_numpy().flatten()
print(f"Dataset cargado: {df_test.shape[0]} filas")

In [ ]:
# Celda 3 - Cargar modelos
print("Cargando modelos...")

# LightGBM
modelo_lgbm = joblib.load(RUTA_MODELO_LGBM)

# MLP (ajusta según cómo lo guardaste)
# Si usaste Keras/TensorFlow:
# from tensorflow.keras.models import load_model
# modelo_mlp = load_model(RUTA_MODELO_MLP)
# Si usaste joblib/sklearn:
modelo_mlp = joblib.load(RUTA_MODELO_MLP)

print("Modelos cargados correctamente")

In [ ]:
# Celda 4 - Preparar features y generar predicciones
# Ajusta esta celda según cómo construiste las features en el entrenamiento

# Ejemplo genérico (reemplaza con tu pipeline real)
feature_cols = [c for c in df_test.columns if c not in ["v1", "target", "label"]]  # ajusta
X_test = df_test.select(feature_cols).to_numpy()

# Predicciones de probabilidad
preds_prob_lgbm = modelo_lgbm.predict_proba(X_test)   # shape (n, 3)
preds_prob_mlp  = modelo_mlp.predict_proba(X_test)    # shape (n, 3)

print(f"Predicciones LGBM: {preds_prob_lgbm.shape}")
print(f"Predicciones MLP : {preds_prob_mlp.shape}")

In [ ]:
# Celda 5 - Función de backtesting
def backtest(preds_prob, precios, ts=0.55, fee=0.00005, k=100, seed=42):
    """
    Realiza backtesting a partir de probabilidades de un modelo.
    Clases: 0=Bajar, 1=Neutral, 2=Subir
    """
    prob_bajar  = preds_prob[:, 0]
    prob_subir  = preds_prob[:, 2]

    # Señales con umbral operativo
    y_pred = np.ones(len(preds_prob), dtype=int)
    y_pred[prob_bajar >= ts] = 0
    y_pred[prob_subir  >= ts] = 2

    n_señales = int((y_pred != 1).sum())
    print(f"  Señales direccionales: {n_señales} ({n_señales/len(y_pred)*100:.2f}%)")

    # Alinear precios (si usaste ventana T en entrenamiento)
    # Si no tienes variable T, usa todos los precios:
    precios_al = precios[-len(y_pred):]   # o precios[T-1:] si corresponde

    assert len(precios_al) == len(y_pred), "Desalineación precios vs predicciones"

    # Retorno futuro
    retorno_futuro = np.zeros(len(precios_al))
    retorno_futuro[:-k] = (precios_al[k:] - precios_al[:-k]) / precios_al[:-k]

    # Beneficios estrategia real
    beneficios = np.zeros(len(precios_al))
    beneficios[y_pred == 2] =  retorno_futuro[y_pred == 2] - fee
    beneficios[y_pred == 0] = -retorno_futuro[y_pred == 0] - fee
    cum_profit = np.cumsum(beneficios)

    # Control aleatorio (shuffle de las señales operativas)
    np.random.seed(seed)
    y_azar = y_pred.copy()
    idx_op = np.where((y_pred == 0) | (y_pred == 2))[0]
    labels_mezclados = y_pred[idx_op].copy()
    np.random.shuffle(labels_mezclados)
    y_azar[idx_op] = labels_mezclados

    beneficios_azar = np.zeros(len(precios_al))
    beneficios_azar[y_azar == 2] =  retorno_futuro[y_azar == 2] - fee
    beneficios_azar[y_azar == 0] = -retorno_futuro[y_azar == 0] - fee
    cum_profit_azar = np.cumsum(beneficios_azar)

    return cum_profit, cum_profit_azar, n_señales

In [ ]:
# Celda 6 - Ejecutar backtesting para ambos modelos
print("="*50)
print("Backtesting LightGBM")
cum_lgbm, cum_azar_lgbm, n_lgbm = backtest(preds_prob_lgbm, precios_completos, ts=ts, fee=fee, k=k)

print("\nBacktesting MLP")
cum_mlp, cum_azar_mlp, n_mlp = backtest(preds_prob_mlp, precios_completos, ts=ts, fee=fee, k=k)

In [ ]:
# Celda 7 - Gráfico comparativo
plt.figure(figsize=(11, 6))

plt.plot(cum_lgbm, color="green", linewidth=2.2, label=f"LightGBM (señales: {n_lgbm})")
plt.plot(cum_mlp,  color="blue",  linewidth=2.2, linestyle="-.", label=f"MLP (señales: {n_mlp})")
plt.plot(cum_azar_lgbm, color="red", linestyle="--", alpha=0.75, label="Azar (control)")

plt.axhline(0, color="black", linestyle=":", alpha=0.5)
plt.title(f"Comparativa de Backtesting  |  ts={ts}  |  fee={fee*100:.4f}%",
          fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Eventos en el Periodo de Test (Ticks)", fontweight="bold")
plt.ylabel("Retorno Acumulado", fontweight="bold")
plt.grid(True, linestyle=":", alpha=0.55)
plt.legend(loc="best", fontsize=10)
plt.tight_layout()

plt.savefig("comparativa_backtesting_mlp_vs_lgbm.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nResultados finales:")
print(f"  LightGBM : {cum_lgbm[-1]:.5f}")
print(f"  MLP      : {cum_mlp[-1]:.5f}")
print(f"  Azar     : {cum_azar_lgbm[-1]:.5f}")